# Week 08 — Home exercise 6: A loader you will call more than once

**Solution proposal.**

The lecture's `load_emissions` could not remove the aggregates, because the information was in
another file. This one can — and it is called twice below with different arguments, which is the
reason it is a function with a parameter rather than a block of cells. Then three questions about
income levels that the lecture, which grouped by region, never asked.

In [1]:
import pandas as pd

## The function

In [2]:
def load_panel(emissions_path, info_path, countries_only=True):
    """
    Read the emissions and country files, join them, and return one table.

    Parameters
    ----------
    emissions_path : str
        Path to the emissions CSV file.
    info_path : str
        Path to the country lookup CSV file, holding one row per entity.
    countries_only : bool, optional
        Whether to drop the World Bank aggregate entities such as World and
        Middle income. Default is True, which is what you want for any total,
        average or ranking across entities. Pass False to keep them, for
        example to check a total you computed against the World Bank's own.

    Returns
    -------
    DataFrame
        One row per entity per year, with co2_pc holding emissions in tonnes
        per person, plus region and income_level from the lookup file.
        Rows with no co2_total are dropped.

    Raises
    ------
    ValueError
        If the merge changes the number of rows, which means the lookup file
        is not one row per entity and every per-entity figure downstream
        would be silently wrong.
    """
    emissions = pd.read_csv(emissions_path)
    emissions = emissions.dropna(subset=["co2_total"])

    # co2_total is in millions of tonnes and population is a count of people,
    # so the ratio is scaled by a million to come out in tonnes per person.
    emissions["co2_pc"] = emissions["co2_total"] * 1_000_000 / emissions["population"]

    info = pd.read_csv(info_path)
    info = info.rename(columns={"incomeLevel": "income_level"})

    # code, not name: two of the names in the lookup file carry a trailing
    # space, and merging on them loses 48 rows without saying anything.
    panel = emissions.merge(
        info[["code", "region", "income_level"]],
        on="code",
        how="left",
        validate="many_to_one",
    )

    if len(panel) != len(emissions):
        raise ValueError(
            f"the merge changed the row count: {len(emissions)} -> {len(panel)}"
        )

    if countries_only:
        panel = panel[panel["region"] != "Aggregates"]

    return panel.reset_index(drop=True)

## Checking both settings

In [3]:
countries = load_panel("../data/co2_emissions.csv", "../data/country_info.csv")
everything = load_panel("../data/co2_emissions.csv", "../data/country_info.csv", countries_only=False)

print("countries only:", countries.shape)
print("everything:    ", everything.shape)
print()
print("entities:", countries["code"].nunique(), "countries out of", everything["code"].nunique())
print()
print(countries["income_level"].value_counts(dropna=False))

countries only: (4872, 14)
everything:     (5904, 14)

entities: 203 countries out of 246

income_level
High income            1848
Upper middle income    1344
Lower middle income    1104
Low income              576
Name: count, dtype: int64


4 872 and 5 904, as expected: 203 countries out of 246 entities. Every country has an income level,
so nothing will silently drop out of the groupings below.

## Who emits?

A share of each year's total needs the year's total beside every row of that year — which is a
`transform`, grouped by **year** rather than by country. The numerator is grouped by year and income
level, the denominator by year alone.

In [4]:
by_level = countries.groupby(["year", "income_level"])["co2_total"].sum().reset_index()

year_total = by_level.groupby("year")["co2_total"].transform("sum")
by_level["share_pct"] = 100 * by_level["co2_total"] / year_total

by_level[by_level["year"].isin([2000, 2010, 2023])].round(1)

,year,income_level,co2_total,share_pct
0,2000,High income,15214.8,62.0
1,2000,Low income,162.4,0.7
2,2000,Lower middle income,1831.7,7.5
3,2000,Upper middle income,7329.7,29.9
40,2010,High income,15351.6,47.3
41,2010,Low income,217.3,0.7
42,2010,Lower middle income,2845.7,8.8
43,2010,Upper middle income,14069.0,43.3
92,2023,High income,13557.4,36.2
93,2023,Low income,222.0,0.6


**In 2000 the high-income countries emitted 62% of the total. By 2023 it was 36%**, and the
upper-middle-income countries had gone from 30% to 51%. Over the same years the high-income countries'
emissions in tonnes barely moved — 15 215 to 13 557 — so the change in shares is almost entirely
growth elsewhere, above all in China.

Two things about the table worth noticing. `reset_index()` straight after the `groupby` gives two
ordinary columns, which is what makes the `transform` on `year` possible. And the low-income
countries, with about a tenth of the world's population, emit well under one percent throughout.

The income levels here are the 2023 classification applied to every year, so the groups hold the same
countries throughout. That is the right choice for following a fixed set of countries over time — but
a country that was lower-middle income in 2000 is counted as upper-middle in 2000 too.

## Checking ourselves against the World Bank

This is what `countries_only=False` is for: the aggregates are the only way to check our own
arithmetic against the bank's.

In [5]:
year_2023 = countries[countries["year"] == 2023]
middle = year_2023[year_2023["income_level"].isin(["Lower middle income", "Upper middle income"])]

ours = middle["co2_total"].sum()
bank = everything[
    (everything["country"] == "Middle income") & (everything["year"] == 2023)
]["co2_total"].iloc[0]

print("middle-income countries:", len(middle))
print(f"our total:               {ours:10,.1f}")
print(f"the bank's Middle income: {bank:9,.1f}")
print(f"difference:              {bank - ours:10,.1f}")

middle-income countries: 102
our total:                 23,703.6
the bank's Middle income:  23,703.6
difference:                     0.0


**They agree exactly.** The bank's `Middle income` figure is the sum of exactly these 102 countries,
and our merge, filter and grouping reproduced it to the last decimal.

That is worth comparing with the lecture's check against `World`, which came out about four percent
short. The difference is informative: `Middle income` is a total over the countries in this file,
and `World` covers territories the file does not. A check that matches exactly tells you the
arithmetic is right; a check that does not tells you to go and find out what the other total
contains.

## Per person, two ways

A group's emissions per person is its total emissions over its total population, so it needs two
columns of the group at once.

**With a function and `.apply()`.** `apply` hands the function each group's rows as a small
DataFrame, and collects one number per group.

In [6]:
def emissions_per_person(group):
    """Total emissions of a group of rows, in tonnes per head of its total population."""
    return group["co2_total"].sum() * 1_000_000 / group["population"].sum()


by_apply = year_2023.groupby("income_level")[["co2_total", "population"]].apply(emissions_per_person)

by_apply.round(2)

income_level
High income            9.80
Low income             0.31
Lower middle income    1.55
Upper middle income    6.32
dtype: float64

**Without a function.** Sum both columns within each group first; the result is a small table with
one row per group, and the division is ordinary column arithmetic on it.

In [7]:
totals = year_2023.groupby("income_level")[["co2_total", "population"]].sum()
by_sum = totals["co2_total"] * 1_000_000 / totals["population"]

print(by_sum.round(2))
print()
print("largest difference between the two:", (by_apply - by_sum).abs().max())

income_level
High income            9.80
Low income             0.31
Lower middle income    1.55
Upper middle income    6.32
dtype: float64

largest difference between the two: 8.881784197001252e-16


The same four numbers, to within the last binary digit of a float.

**Which to write?** The second. It uses only built-in aggregations, so pandas does each sum in one
vectorized pass, and it leaves the intermediate totals in a table you can look at. The `apply`
version calls a Python function once per group — four times here, which costs nothing, but forty
thousand times on a bigger grouping, which does.

What `apply` buys is generality. It earns its place when the calculation cannot be split into
separate column aggregations followed by arithmetic — a weighted median, say, or anything that needs
to sort the group's rows first. A ratio of two sums can always be split, so here it is the longer way
to the same place.

## Weighted against unweighted

In [8]:
comparison = pd.DataFrame({
    "per_person": by_sum,
    "mean_of_countries": year_2023.groupby("income_level")["co2_pc"].mean(),
    "countries": year_2023.groupby("income_level")["co2_pc"].count(),
})
comparison["difference"] = comparison["per_person"] - comparison["mean_of_countries"]

comparison.round(2)

,per_person,mean_of_countries,countries,difference
income_level,,,,
High income,9.80,8.55,77,1.25
Low income,0.31,0.33,24,-0.02
Lower middle income,1.55,1.08,46,0.47
Upper middle income,6.32,3.53,56,2.78


The two agree for low-income countries, and disagree most for **upper-middle income: 6.32 tonnes per
person against an unweighted mean of 3.53.**

In [9]:
upper = year_2023[year_2023["income_level"] == "Upper middle income"]

upper.sort_values("population", ascending=False)[["country", "population", "co2_total", "co2_pc"]].head(5).round(2)

,country,population,co2_total,co2_pc
959,China,1410710000,13021.24,9.23
2111,Indonesia,281190067,772.56,2.75
623,Brazil,211140729,480.24,2.27
2927,Mexico,129739759,475.60,3.67
3551,Philippines,114891199,162.45,1.41


In [10]:
china = upper[upper["country"] == "China"]
without_china = upper[upper["country"] != "China"]

print(f"China's share of the group's population: {china['population'].sum() / upper['population'].sum():.0%}")
print(f"China's share of the group's emissions:  {china['co2_total'].sum() / upper['co2_total'].sum():.0%}")
print()
print(f"per person without China: {without_china['co2_total'].sum() * 1_000_000 / without_china['population'].sum():.2f}")
print(f"mean without China:       {without_china['co2_pc'].mean():.2f}")

China's share of the group's population: 46%
China's share of the group's emissions:  68%

per person without China: 3.81
mean without China:       3.43


**China is 46% of the group's people and 68% of its emissions**, at 9.23 tonnes per person — more than
twice the group's typical country. The weighted figure gives China almost half the weight, so it is
pulled up towards China's level. The unweighted mean gives China one vote out of 56, so it barely
notices.

Take China out and the two measures almost meet, at 3.81 and 3.43. The disagreement was never about
upper-middle-income countries in general. It was about one of them.

The other two gaps have the same structure. The high-income weighted figure is pulled up by the United
States, the largest high-income population, at 13.7 tonnes per person; the lower-middle one by India,
which holds half the group's people and emits more per person than most of the other lower-middle
countries.

The rule underneath, as in the lecture: **the unweighted mean answers "what does a typical country in
this group look like?" and the weighted one answers "how much does a typical person in this group
emit?"** When one country is half the group, those are very different questions.

### Things worth noticing

- **The function raises rather than returning something wrong.** A comment saying "the merge should
  not change the row count" is a hope; `raise ValueError` is a check. It costs three lines and it
  converts the worst failure mode of this week — a silently reshaped table — into a stack trace at the
  point of the mistake.
- **`validate="many_to_one"` and the row-count check overlap, and both are worth having.** `validate`
  catches duplicate keys in the lookup; the count check also catches anything else that changes the
  shape, including a future edit to this function.
- **The docstring documents what it raises**, because a caller needs to know that this function can
  fail loudly. A `Raises` section is part of the interface, not decoration.
- **`transform` works on any grouping, not only the one the rows are a series within.** Grouping by
  year to put each year's total beside every row is the same move as grouping by country to put each
  country's average there.

### What this notebook does NOT do

- **It trusts the lookup file's `region` column completely.** Every country in the output is a country
  because that file says so. If an entity were missing from the lookup, `region` would be `NaN`,
  `NaN != "Aggregates"` is `True`, and it would be quietly kept as a country — the filter would not
  catch it. A stricter version would check `panel["region"].isna().sum() == 0` too.
- **It does not check the columns it depends on exist.** A renamed column in a future download raises a
  `KeyError` from somewhere in the middle of the function rather than a message saying which file was
  the wrong shape.
- **It uses one income classification for every year.** The shares over time follow a fixed set of
  countries, which is a choice; the World Bank's own history of classifications would tell a
  different story about which group grew.
- It has no tests. Nothing checks that `load_panel` on a file with a duplicated code actually raises,
  which is the one behavior the function most needs to be right about.